<div class="alert alert-block alert-success">
    <h1 align="center">Hand Frame Effect | MediaPipe & OpenCV</h1>
    <h3 align="center">Real-time Hand Gesture Controlled Visual Effects</h3>
    
<a href="https://www.instagram.com/p/DdCHJ6GO9m8/">
Original Video
    </a>
</div>

<img src="https://images.pexels.com/photos/29194060/pexels-photo-29194060.jpeg?_gl=1*10ihzq*_ga*MTEzOTgwNjM4MS4xNzY5MTc0MDY2*_ga_8JE65Q40S6*czE3ODkwNjY4ODMkbzMkZzEkdDE3ODkwNjY5NjAkajU1JGwwJGgw" 
     style="width: 100%; max-width: 1200px; height: 380px; object-fit: cover; display: block; margin: 0 auto; border-radius: 10px;">



<div style="
        background-color:#e6f9e6;
        border-left:6px solid #4caf50;
        padding: 20px;
        border-radius: 8px;
        margin: 20px 0;
        font-size: 16px;
        line-height: 1.6; ">
    <h2 style="text-align: center; color: #007400; margin-top: 0;">
        Welcome to the Notebook...! 
    </h2>
    <p style="text-align: justify; margin-bottom: 20px;">
Given the recent popularity of this trend on TikTok and Instagram, where creators form a rectangle with their hands and apply real-time visual effects inside it using computer vision, we are going to recreate and implement this interactive effect from scratch.
    </p>
<p style="text-align: justify; margin-bottom: 20px;">
In this notebook, we will build a Real-time Hand Gesture Controlled Visual Effects system. The idea is simple yet impressive: the user forms a rectangular frame with both hands in front of the camera, and only the region inside that frame receives live visual effects such as sharpening, black & white, pixelation, neon outlines, and more. Additionally, by performing a pinch gesture (closing and opening the thumb and index finger), the applied effect can be switched dynamically.
</p>
    </p>
<p style="text-align: justify; margin-bottom: 20px;">
To achieve this, we will leverage the following techniques and libraries:
</p>
    <ul style="margin: 0; padding-left: 20px; text-align: justify;">
            <li>
            Hand Landmark Detection using MediaPipe for precise real-time tracking of finger positions
         </li>
         <li>
             Gesture Recognition to detect the framing pose and pinch action
         </li>
         <li>
             Dynamic Region of Interest (ROI) Filtering to apply effects only within the hand-defined area
         </li>
         <li>
             Real-time Image Processing and Compositing with OpenCV
         </li>
        </ul>
</div>

<div style="background-color:#e6f9e6;border-left:6px solid #4caf50;padding:15px 20px;border-radius:8px;margin:15px 0;font-size:16px;line-height:1.5;">
    <p style="font-weight:bold;margin-bottom:10px;">
        Note:
    </p>
        </p>
<p style="text-align: justify; margin-bottom: 20px;">
Due to the limitations of the Kaggle coding environment, we cannot open interactive OpenCV windows or access a live webcam feed. Therefore, instead of real-time interaction, this notebook demonstrates the final processed output using sample images/videos.
</p>
</div>

<div style="background-color:#e6f9e6;border-left:6px solid #4caf50;padding:15px 20px;border-radius:8px;margin:15px 0;font-size:16px;line-height:1.5;">
    <p style="font-weight:bold;margin-bottom:10px;">
        Install required libraries
    </p>
</div>

In [ ]:
#!pip install opencv-python mediapipe

<div style="background-color:#e6f9e6;border-left:6px solid #4caf50;padding:15px 20px;border-radius:8px;margin:15px 0;font-size:16px;line-height:1.5;">
    <p style="font-weight:bold;margin-bottom:10px;">
        Setup and download Hand Landmarker model
    </p>
</div>

In [ ]:
from IPython.display import HTML, display, Video
#---
import urllib.request
import os
import time
#--
import cv2
import numpy as np
#---
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# Download the Hand Landmarker model if it is not already available
MODEL_PATH = "hand_landmarker.task"
if not os.path.exists(MODEL_PATH):
    print("Downloading Hand Landmarker model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        MODEL_PATH
    )
    print("Model downloaded.")

<div style="background-color:#e6f9e6;border-left:6px solid #4caf50;padding:15px 20px;border-radius:8px;margin:15px 0;font-size:16px;line-height:1.5;">
    <p style="font-weight:bold;margin-bottom:10px;">
       Hand-Selected Image Effects
    </p>
</div>

In [ ]:
#Effects:
def sharpen(img):
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32)
    return cv2.filter2D(img, -1, kernel)
#---
def sharpen_strong(img):
    kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]], np.float32)
    return cv2.filter2D(img, -1, kernel)
#---
def neon(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    colored = cv2.applyColorMap(edges, cv2.COLORMAP_PLASMA)
    return cv2.addWeighted(img, 0.4, colored, 0.9, 0)
#---
def black_white(img):
    return cv2.cvtColor(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)
#---
def invert(img):
    return 255 - img
#---
def pixelate(img):
    h, w = img.shape[:2]
    small = cv2.resize(img, (12, 12), interpolation=cv2.INTER_LINEAR)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
#---
def warm(img):
    result = img.astype(np.float32)
    result[:, :, 2] *= 1.25   # Red
    result[:, :, 1] *= 1.08   # Green
    result[:, :, 0] *= 0.85   # Blue

    return np.clip(result, 0, 255).astype(np.uint8)
#---
def cold(img):
    result = img.astype(np.float32)
    result[:, :, 0] *= 1.30   # Blue
    result[:, :, 1] *= 1.05
    result[:, :, 2] *= 0.80   # Red

    return np.clip(result, 0, 255).astype(np.uint8)
#---
def cartoon(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 5)
    edges = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 9 )
    color = cv2.bilateralFilter(img, 9, 75, 75)
    
    return cv2.bitwise_and(color, color, mask=edges   )
#---
def sketch(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    inverted = 255 - gray
    blur = cv2.GaussianBlur(inverted, (21, 21), 0)
    sketch_img = cv2.divide(gray, 255 - blur, scale=256)

    return cv2.cvtColor(sketch_img,cv2.COLOR_GRAY2BGR)
#---
def vintage(img):
    result = img.astype(np.float32)
    result[:, :, 0] *= 0.75
    result[:, :, 1] *= 0.95
    result[:, :, 2] *= 1.15
    result = np.clip(result, 0, 255).astype(np.uint8)
    result = cv2.GaussianBlur(result, (3, 3), 0)

    return result
#---
def edge_glow(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 80, 180)
    edges = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)
    glow = cv2.GaussianBlur(edges, (21, 21), 0)
    glow = cv2.cvtColor(glow, cv2.COLOR_GRAY2BGR)

    return cv2.addWeighted(img, 0.7, glow, 1.5, 0)
#---

EFFECTS = [
    ("Sharpen", sharpen),
    ("Sharpen Strong", sharpen_strong),
    ("Neon", neon),
    ("Black & White", black_white),
    ("Invert", invert),
    ("Pixelate", pixelate),
    ("Warm", warm),
    ("Cold", cold),
    ("Cartoon", cartoon),
    ("Sketch", sketch),
    ("Vintage", vintage),
    ("Edge Glow", edge_glow),
]

def main():
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=2,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3
    )
    detector = vision.HandLandmarker.create_from_options(options)

    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    effect_idx = 0
    smoothed_pts = None
    last_pinch_time = 0
    pinch_cooldown = 0.8
    debug_mode = False

    print("Controls: SPACE/N = change effect | D = debug mode | Q = quit")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]
        output = frame.copy()

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = detector.detect(mp_image)

        num_hands = len(result.hand_landmarks) if result.hand_landmarks else 0
        cv2.putText(output, f"Hands: {num_hands}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        all_points = []
        pinch_detected = False

        if result.hand_landmarks:
            for hand_landmarks in result.hand_landmarks:
                for lm in hand_landmarks:
                    x, y = int(lm.x * w), int(lm.y * h)
                    cv2.circle(output, (x, y), 4, (0, 255, 0), -1)

                thumb = hand_landmarks[4]
                index = hand_landmarks[8]
                tx, ty = int(thumb.x * w), int(thumb.y * h)
                ix, iy = int(index.x * w), int(index.y * h)

                all_points.append([tx, ty])
                all_points.append([ix, iy])

                dist = np.hypot(tx - ix, ty - iy)
                if dist < 40:
                    pinch_detected = True

        if pinch_detected and (time.time() - last_pinch_time) > pinch_cooldown:
            effect_idx = (effect_idx + 1) % len(EFFECTS)
            last_pinch_time = time.time()
            print("Effect changed →", EFFECTS[effect_idx][0])

        if len(all_points) >= 4:
            pts = np.array(all_points[:4], dtype=np.float32)
            hull = cv2.convexHull(pts).reshape(-1, 2)

            if smoothed_pts is None or len(smoothed_pts) != len(hull):
                smoothed_pts = hull.copy()
            else:
                smoothed_pts = 0.6 * smoothed_pts + 0.4 * hull

            pts_i = np.round(smoothed_pts).astype(np.int32)

            mask = np.zeros((h, w), dtype=np.uint8)
            cv2.fillConvexPoly(mask, pts_i, 255)

            if cv2.countNonZero(mask) > 500:
                if debug_mode:
                    effected = np.zeros_like(frame)
                    effected[:] = (0, 0, 255)
                else:
                    name, fn = EFFECTS[effect_idx]
                    effected = fn(frame)

                mask_inv = cv2.bitwise_not(mask)
                bg = cv2.bitwise_and(frame, frame, mask=mask_inv)
                fg = cv2.bitwise_and(effected, effected, mask=mask)
                output = cv2.add(bg, fg)

                cv2.polylines(output, [pts_i], True, (255, 255, 255), 5)
                for p in pts_i:
                    cv2.circle(output, tuple(p), 10, (255, 255, 255), -1)

                cv2.putText(output, f"Effect: {EFFECTS[effect_idx][0]}", (20, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 255), 2)
            else:
                smoothed_pts = None
        else:
            smoothed_pts = None
            cv2.putText(output, "Make rectangle with BOTH hands", (20, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 140, 255), 2)

        if debug_mode:
            cv2.putText(output, "DEBUG MODE (Red inside)", (20, 140),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        cv2.imshow("Hand Frame Effect", output)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break
        elif key == ord(' ') or key == ord('n'):
            effect_idx = (effect_idx + 1) % len(EFFECTS)
            print("Effect →", EFFECTS[effect_idx][0])
        elif key == ord('d'):
            debug_mode = not debug_mode
            print("Debug mode:", debug_mode)

    cap.release()
    cv2.destroyAllWindows()
    detector.close()

if __name__ == "__main__":
    main()

<div style="background-color:#e6f9e6;border-left:6px solid #4caf50;padding:15px 20px;border-radius:8px;margin:15px 0;font-size:16px;line-height:1.5;">
    <p style="font-weight:bold;margin-bottom:10px;">
       Result
    </p>
</div>

In [1]:
Video(result, embed=True, width=800, height=500)

<div style="background-color:#e6f9e6;border-left:6px solid #4caf50;padding:15px 20px;border-radius:8px;margin:15px 0;font-size:16px;line-height:1.5;">
    <p style="font-weight:bold;margin-bottom:10px;">
        Thanks a lot for going through this notebook! I hope it was helpful and worth your time.
    </p>
</div>